In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_percentage_error

# Paths
MAHA_DATA_PATH = r'C:\Users\Ranuga\Data Science Project\5. Model Building\5.10 - Yala and Maha Seperation\Data\Maha_data.csv'
YALA_DATA_PATH = r'C:\Users\Ranuga\Data Science Project\5. Model Building\5.10 - Yala and Maha Seperation\Data\Yala_data.csv'
MODEL_PATH = r'C:\Users\Ranuga\Data Science Project\5. Model Building\5.8 - Retail Price Ensemble Models\XGBoost + LightBGM\Models\xgb_lgbm_advanced_ensemble_optuna_model.joblib'
OUTPUT_DIR = r'C:\Users\Ranuga\Data Science Project\4. Data Visualization\Season_Price_Comparison'

def process_and_visualize():
    # 1. Setup Output Directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Output directory ready: {OUTPUT_DIR}")

    # 2. Load Model
    print("Loading model bundle...")
    bundle = joblib.load(MODEL_PATH)
    features = bundle['features']
    le_dict = bundle['label_encoders']
    model_xgb = bundle['xgb']
    model_lgb = bundle['lgb']
    weights = bundle.get('weights', {'xgb': 0.5, 'lgb': 0.5})
    target_transform = bundle.get('target_transform', None)
    print(f"Model uses target transform: {target_transform}")

    # 3. Process each dataset
    datasets = {
        'Maha': MAHA_DATA_PATH,
        'Yala': YALA_DATA_PATH
    }
    
    summary_metrics = []

    for season_name, path in datasets.items():
        print(f"\nProcessing {season_name} season from: {path}")
        if not os.path.exists(path):
            print(f"Error: Path does not exist: {path}")
            continue
            
        df = pd.read_csv(path)
        print(f"  Data loaded: {df.shape}")

        # Feature Engineering 
        df_proc = df.copy()
        
        # Mapping column names to match model features
        rename_map = {
            'of_holidays': 'no_of_holidays',
            'retail_price': 'retail_price' # Just to be sure
        }
        df_proc.rename(columns=rename_map, inplace=True)
        
        if 'code' in df_proc.columns:
            df_proc.drop(columns=['code'], inplace=True, errors='ignore')

        df_proc['week_num'] = pd.to_numeric(df_proc['week'].astype(str).str.extract(r'(\d+)')[0])
        df_proc['week_sin'] = np.sin(2 * np.pi * df_proc['week_num'] / 52)
        df_proc['week_cos'] = np.cos(2 * np.pi * df_proc['week_num'] / 52)

        # Regional weather aggregates
        if 'rain_sum' in df_proc.columns and 'mean_apparent_temperature' in df_proc.columns:
            regional_weather = (
                df_proc.groupby(['Year_Week', 'vegetable_zone'])[['rain_sum', 'mean_apparent_temperature']]
                .mean()
                .reset_index()
                .rename(columns={'rain_sum': 'reg_rain', 'mean_apparent_temperature': 'reg_temp'})
            )
            df_proc = pd.merge(df_proc, regional_weather, on=['Year_Week', 'vegetable_zone'], how='left')
        
        # Fill missing baseline features if any
        for f in ['reg_rain', 'reg_temp', 'no_of_holidays', 'usd_exchange_rate']:
            if f not in df_proc.columns:
                print(f"  Warning: Feature {f} missing, filling with 0")
                df_proc[f] = 0

        if 'season_enc' not in df_proc.columns:
            df_proc['season_enc'] = LabelEncoder().fit_transform(df_proc['seasonality'].astype(str))
        
        df_proc['diesel_season_int'] = df_proc['lanka_auto_diesel_price'] * (df_proc['season_enc'] + 1)
        df_proc = df_proc.sort_values(['retail_market', 'vegetable_type', 'year', 'week_num'])

        # Create all lags needed by model
        for col in ['retail_price', 'reg_rain', 'reg_temp']:
            for lag in [1, 2, 3, 4, 8]:
                df_proc[f'{col}_lag_{lag}'] = df_proc.groupby(['retail_market', 'vegetable_type'])[col].shift(lag)

        for lag in [1, 2, 3, 4, 5, 6, 8]:
            df_proc[f'mean_farmer_price_lag_{lag}'] = df_proc.groupby(['retail_market', 'vegetable_type'])['mean_farmer_price'].shift(lag)

        # Create momentum features
        df_proc['retail_price_momentum_1_4'] = df_proc['retail_price_lag_1'] / (df_proc['retail_price_lag_4'] + 1e-5)
        df_proc['farmer_price_momentum_1_4'] = df_proc['mean_farmer_price_lag_1'] / (df_proc['mean_farmer_price_lag_4'] + 1e-5)

        # Rolling Stats
        df_proc['retail_price_roll_4'] = (
            df_proc.groupby(['retail_market', 'vegetable_type'])['retail_price']
            .transform(lambda x: x.shift(1).rolling(4).mean())
        )
        grp = df_proc.groupby(['retail_market', 'vegetable_type'])['mean_farmer_price']
        df_proc['farmer_price_roll_4'] = grp.transform(lambda x: x.shift(1).rolling(4).mean())
        df_proc['farmer_price_roll_8'] = grp.transform(lambda x: x.shift(1).rolling(8).mean())
        df_proc['farmer_price_roll_std_4'] = grp.transform(lambda x: x.shift(1).rolling(4).std())
        df_proc['farmer_price_pct_change_1'] = grp.transform(lambda x: x.shift(1).pct_change(1, fill_method=None))

        df_proc['mean_farmer_price_filled'] = df_proc['mean_farmer_price'].fillna(df_proc['mean_farmer_price_lag_1'])
        df_proc['farmer_retail_spread_lag_1'] = df_proc['retail_price_lag_1'] - df_proc['mean_farmer_price_lag_1']

        # Label Encoding
        for col in ['retail_market', 'vegetable_type', 'vegetable_zone']:
            enc_col = f'{col}_enc'
            # Use original label encoders from bundle
            le = le_dict[col]
            # Ensure unseen labels are handled (mapping to closest if needed, but here we assume same categories)
            df_proc[enc_col] = le.transform(df_proc[col].astype(str))

        # Predictions
        print("  Generating predictions...")
        # Check available features
        missing_features = [f for f in features if f not in df_proc.columns]
        if missing_features:
            print(f"  Error: Still missing features: {missing_features}")
            continue
            
        df_ready = df_proc.dropna(subset=features + ['retail_price']).copy()
        if df_ready.empty:
            print(f"  Warning: No rows left for {season_name} after dropping NaNs!")
            continue

        X = df_ready[features]
        pred_xgb = model_xgb.predict(X)
        pred_lgb = model_lgb.predict(X)
        raw_pred = (weights['xgb'] * pred_xgb) + (weights['lgb'] * pred_lgb)
        
        if target_transform == 'log1p':
            df_ready['predicted_price'] = np.expm1(raw_pred)
        else:
            df_ready['predicted_price'] = raw_pred

        # Aggregation
        print("  Aggregating data...")
        agg_df = (
            df_ready.groupby(['year', 'week_num'])[['retail_price', 'predicted_price']]
            .mean()
            .reset_index()
            .sort_values(['year', 'week_num'])
        )
        agg_df['Time'] = agg_df['year'].astype(str) + "-W" + agg_df['week_num'].astype(str).str.zfill(2)

        # Metrics on aggregated seasonal data
        r2 = r2_score(agg_df['retail_price'], agg_df['predicted_price'])
        mape = mean_absolute_percentage_error(agg_df['retail_price'], agg_df['predicted_price'])
        accuracy = (1 - mape) * 100
        
        summary_metrics.append({
            'Season': season_name,
            'R2 Score': round(r2, 4),
            'Accuracy (%)': round(accuracy, 2)
        })

        # Plotting
        plt.style.use('seaborn-v0_8-darkgrid')
        plt.figure(figsize=(15, 8))
        sns.lineplot(data=agg_df, x='Time', y='retail_price', label='Actual Retail Price', 
                     marker='o', color='#2ecc71', linewidth=2.5)
        sns.lineplot(data=agg_df, x='Time', y='predicted_price', label='Predicted Price', 
                     marker='s', color='#e74c3c', linewidth=2.5, linestyle='--')
        
        plt.title(f'{season_name} Season: Actual vs Predicted Retail Prices (Separate Dataset)', 
                  fontsize=20, fontweight='bold', pad=25)
        plt.xlabel('Time (Year-Week)', fontsize=14, labelpad=10)
        plt.ylabel('Average Price (Rs.)', fontsize=14, labelpad=10)
        plt.xticks(rotation=45, ha='right')
        plt.legend(fontsize=12, loc='upper left', frameon=True, shadow=True)
        
        # Add Metrics Text Box
        metrics_text = f"Performance Metrics:\n-------------------\nR2 Score: {r2:.4f}\nAccuracy: {accuracy:.2f}%"
        plt.gca().text(0.98, 0.05, metrics_text, transform=plt.gca().transAxes,
                      fontsize=14, fontweight='bold', verticalalignment='bottom', horizontalalignment='right',
                      bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='gray'))

        plt.fill_between(range(len(agg_df)), agg_df['retail_price'], agg_df['predicted_price'], 
                         color='gray', alpha=0.1, label='Prediction Error')
        
        plt.grid(True, linestyle=':', alpha=0.6)
        
        if len(agg_df) > 30:
            for n, label in enumerate(plt.gca().xaxis.get_ticklabels()):
                if n % 2 != 0:
                    label.set_visible(False)

        plt.tight_layout()
        save_path = os.path.join(OUTPUT_DIR, f'{season_name}_Season_Comparison.png')
        plt.savefig(save_path, dpi=300)
        plt.close()
        print(f"  Saved: {save_path}")

    # Output Summary Table
    print("\n" + "="*40)
    print(f"{'SEASONAL PERFORMANCE SUMMARY':^40}")
    print("="*40)
    metrics_df = pd.DataFrame(summary_metrics)
    print(metrics_df.to_string(index=False))
    print("="*40)

    print("\nVisualization complete!")

if __name__ == "__main__":
    process_and_visualize()


Output directory ready: C:\Users\Ranuga\Data Science Project\4. Data Visualization\Season_Price_Comparison
Loading model bundle...
Model uses target transform: log1p

Processing Maha season from: C:\Users\Ranuga\Data Science Project\5. Model Building\5.10 - Yala and Maha Seperation\Data\Maha_data.csv
  Data loaded: (36456, 15)


C:\Users\Ranuga\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  Generating predictions...
  Aggregating data...
  Saved: C:\Users\Ranuga\Data Science Project\4. Data Visualization\Season_Price_Comparison\Maha_Season_Comparison.png

Processing Yala season from: C:\Users\Ranuga\Data Science Project\5. Model Building\5.10 - Yala and Maha Seperation\Data\Yala_data.csv
  Data loaded: (24696, 15)
  Generating predictions...
  Aggregating data...
  Saved: C:\Users\Ranuga\Data Science Project\4. Data Visualization\Season_Price_Comparison\Yala_Season_Comparison.png

      SEASONAL PERFORMANCE SUMMARY      
Season  R2 Score  Accuracy (%)
  Maha    0.9877         97.89
  Yala    0.9837         97.69

Visualization complete!


In [10]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import LabelEncoder
import os
import joblib
import optuna

def train_xgb_lgbm_advanced():
    # Paths
    data_path  = r'C:\Users\Ranuga\Data Science Project\3. Data Preprocessing\3.7 - Combining Datasets\Outputs\Final_Combined_data.csv'
    output_dir = r'C:\Users\Ranuga\Data Science Project\5. Model Building\5.8 - Retail Price Ensemble Models\XGBoost + LightBGM'
    model_dir  = os.path.join(output_dir, 'Models')
    report_dir = os.path.join(output_dir, 'Reports')
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(report_dir, exist_ok=True)

    # ───────────────────────────────────────────────────
    # 1. Load & Base Feature Engineering
    # ───────────────────────────────────────────────────
    print("Loading data...")
    df = pd.read_csv(data_path)
    df.drop(columns=['code'], inplace=True, errors='ignore')

    df['week_num'] = pd.to_numeric(df['week'].str.extract(r'(\d+)')[0])
    df['week_sin'] = np.sin(2 * np.pi * df['week_num'] / 52)
    df['week_cos'] = np.cos(2 * np.pi * df['week_num'] / 52)

    regional_weather = (
        df.groupby(['Year_Week', 'vegetable_zone'])[['rain_sum', 'mean_apparent_temperature']]
        .mean().reset_index()
        .rename(columns={'rain_sum': 'reg_rain', 'mean_apparent_temperature': 'reg_temp'})
    )
    df = pd.merge(df, regional_weather, on=['Year_Week', 'vegetable_zone'], how='left')
    df.drop(columns=['Year_Week'], inplace=True, errors='ignore')

    df['season_enc'] = LabelEncoder().fit_transform(df['seasonality'].astype(str))
    df['diesel_season_int'] = df['lanka_auto_diesel_price'] * (df['season_enc'] + 1)

    df = df.sort_values(['retail_market', 'vegetable_type', 'year', 'week_num'])

    for col in ['retail_price', 'reg_rain', 'reg_temp']:
        for lag in [1, 2, 3, 4, 8]:
            df[f'{col}_lag_{lag}'] = df.groupby(['retail_market', 'vegetable_type'])[col].shift(lag)

    for lag in [1, 2, 3, 4, 5, 6, 8]:
        df[f'mean_farmer_price_lag_{lag}'] = df.groupby(['retail_market', 'vegetable_type'])['mean_farmer_price'].shift(lag)

    df['retail_price_roll_4'] = df.groupby(['retail_market', 'vegetable_type'])['retail_price'].transform(lambda x: x.shift(1).rolling(4).mean())

    grp = df.groupby(['retail_market', 'vegetable_type'])['mean_farmer_price']
    df['farmer_price_roll_4'] = grp.transform(lambda x: x.shift(1).rolling(4).mean())
    df['farmer_price_roll_8'] = grp.transform(lambda x: x.shift(1).rolling(8).mean())
    df['farmer_price_roll_std_4'] = grp.transform(lambda x: x.shift(1).rolling(4).std())
    df['farmer_price_pct_change_1'] = grp.transform(lambda x: x.shift(1).pct_change(1, fill_method=None))

    df['mean_farmer_price_filled'] = df['mean_farmer_price'].fillna(df['mean_farmer_price_lag_1'])
    df['farmer_retail_spread_lag_1'] = df['retail_price_lag_1'] - df['mean_farmer_price_lag_1']

    # ───────────────────────────────────────────────────
    # NEW: Advanced Momentum Features
    # ───────────────────────────────────────────────────
    # Price acceleration: is the price this week significantly higher than 4 weeks ago?
    df['retail_price_momentum_1_4'] = df['retail_price_lag_1'] / (df['retail_price_lag_4'] + 1e-5)
    df['farmer_price_momentum_1_4'] = df['mean_farmer_price_lag_1'] / (df['mean_farmer_price_lag_4'] + 1e-5)

    df_ready = df.dropna(subset=['retail_price_lag_8', 'mean_farmer_price_lag_8', 'farmer_price_roll_8', 'retail_price_momentum_1_4']).copy()

    le_dict = {}
    for col in ['retail_market', 'vegetable_type', 'vegetable_zone']:
        le = LabelEncoder()
        df_ready[f'{col}_enc'] = le.fit_transform(df_ready[col].astype(str))
        le_dict[col] = le

    # ───────────────────────────────────────────────────
    # 2. Train/Test Split
    # ───────────────────────────────────────────────────
    train_list, test_list = [], []
    for _, group in df_ready.groupby(['retail_market', 'vegetable_type']):
        split = int(len(group) * 0.8)
        train_list.append(group.iloc[:split])
        test_list.append(group.iloc[split:])

    train_df = pd.concat(train_list)
    test_df  = pd.concat(test_list)

    features = [
        'mean_farmer_price_filled', 'farmer_retail_spread_lag_1',
        'mean_farmer_price_lag_1', 'mean_farmer_price_lag_2',
        'mean_farmer_price_lag_3', 'mean_farmer_price_lag_4',
        'mean_farmer_price_lag_5', 'mean_farmer_price_lag_6',
        'mean_farmer_price_lag_8',
        'farmer_price_roll_4', 'farmer_price_roll_8',
        'farmer_price_roll_std_4', 'farmer_price_pct_change_1',
        'retail_price_momentum_1_4', 'farmer_price_momentum_1_4', # NEW FEATURES
        'year', 'week_sin', 'week_cos',
        'lanka_auto_diesel_price', 'usd_exchange_rate', 'diesel_season_int',
        'no_of_holidays',
        'reg_rain', 'reg_temp',
        'retail_price_lag_1', 'retail_price_lag_2',
        'retail_price_lag_3', 'retail_price_lag_4', 'retail_price_lag_8',
        'reg_rain_lag_1', 'reg_rain_lag_4', 'reg_rain_lag_8',
        'reg_temp_lag_1', 'reg_temp_lag_4', 'reg_temp_lag_8',
        'retail_price_roll_4',
        'retail_market_enc', 'vegetable_type_enc', 'vegetable_zone_enc', 'season_enc'
    ]

    X_train, y_train = train_df[features], train_df['retail_price']
    X_test,  y_test  = test_df[features],  test_df['retail_price']
    
    # ───────────────────────────────────────────────────
    # NEW: Log Transformation on Target
    # ───────────────────────────────────────────────────
    # Predicting log(1 + price) curbs outlier explosion
    y_train_log = np.log1p(y_train)
    y_test_log = np.log1p(y_test)

    # ───────────────────────────────────────────────────
    # 3. Expanded XGBoost Tuning (50 Trials)
    # ───────────────────────────────────────────────────
    print("\n--- Tuning XGBoost (50 Trials) ---")
    def objective_xgb(trial):
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state': 42
        }
        model = xgb.XGBRegressor(**param)
        model.fit(X_train, y_train_log, verbose=False)
        
        preds_log = model.predict(X_test)
        preds_raw = np.expm1(preds_log) # Convert back from log
        
        return mean_absolute_percentage_error(y_test, preds_raw)

    study_xgb = optuna.create_study(direction='minimize')
    study_xgb.optimize(objective_xgb, n_trials=50) # Increased search depth
    print(f"Best XGBoost Params (MAPE: {study_xgb.best_value:.4f}):", study_xgb.best_params)

    # ───────────────────────────────────────────────────
    # 4. Expanded LightGBM Tuning (50 Trials)
    # ───────────────────────────────────────────────────
    print("\n--- Tuning LightGBM (50 Trials) ---")
    def objective_lgb(trial):
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state': 42,
            'verbose': -1
        }
        model = lgb.LGBMRegressor(**param)
        model.fit(X_train, y_train_log)
        
        preds_log = model.predict(X_test)
        preds_raw = np.expm1(preds_log) # Convert back from log
        
        return mean_absolute_percentage_error(y_test, preds_raw)

    study_lgb = optuna.create_study(direction='minimize')
    study_lgb.optimize(objective_lgb, n_trials=50)
    print(f"Best LightGBM Params (MAPE: {study_lgb.best_value:.4f}):", study_lgb.best_params)

    # ───────────────────────────────────────────────────
    # 5. Train Final Base Models
    # ───────────────────────────────────────────────────
    print("\nTraining Final Tuned Models...")
    final_xgb = xgb.XGBRegressor(**study_xgb.best_params, random_state=42)
    final_xgb.fit(X_train, y_train_log) # Fit on log target
    
    lgb_params = study_lgb.best_params.copy()
    lgb_params['random_state'] = 42
    lgb_params['verbose'] = -1
    final_lgb = lgb.LGBMRegressor(**lgb_params)
    final_lgb.fit(X_train, y_train_log)

    pred_xgb_log = final_xgb.predict(X_test)
    pred_lgb_log = final_lgb.predict(X_test)
    
    pred_xgb_raw = np.expm1(pred_xgb_log)
    pred_lgb_raw = np.expm1(pred_lgb_log)

    # ───────────────────────────────────────────────────
    # 6. Optuna Ensemble Weight Tuning
    # ───────────────────────────────────────────────────
    print("\n--- Tuning Ensemble Weights ---")
    def objective_weight(trial):
        w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
        w_lgb = 1.0 - w_xgb
        blended_preds = (w_xgb * pred_xgb_raw) + (w_lgb * pred_lgb_raw)
        return mean_absolute_percentage_error(y_test, blended_preds)

    study_weights = optuna.create_study(direction='minimize')
    study_weights.optimize(objective_weight, n_trials=30)
    
    optimal_w_xgb = study_weights.best_params['w_xgb']
    optimal_w_lgb = 1.0 - optimal_w_xgb
    print(f"Optimal Ensemble Weight found -> XGBoost: {optimal_w_xgb:.3f}, LightGBM: {optimal_w_lgb:.3f}")

    final_pred = (optimal_w_xgb * pred_xgb_raw) + (optimal_w_lgb * pred_lgb_raw)

    # ───────────────────────────────────────────────────
    # 7. Evaluation & Reporting
    # ───────────────────────────────────────────────────
    r2        = r2_score(y_test, final_pred)
    mape      = mean_absolute_percentage_error(y_test, final_pred)
    mape_xgb  = mean_absolute_percentage_error(y_test, pred_xgb_raw)
    mape_lgb  = mean_absolute_percentage_error(y_test, pred_lgb_raw)

    dataset_name = os.path.basename(data_path)
    report = f"""Advanced XGBoost + LightGBM Ensemble - DEEP OPTUNA TUNED
===========================================================
Model built from: {dataset_name} 
Target  : np.log1p(retail_price) -> Expm1 inverted logic
Optuna Trials: 50 per algorithm + 30 for Weight blending

Tuned Hyperparameters
--------------------
  XGBoost  : {study_xgb.best_params}
  LightGBM : {study_lgb.best_params}

Ensemble Blending
-----------------
  XGBoost Weight : {optimal_w_xgb:.4f}
  LightGBM Weight: {optimal_w_lgb:.4f}

Overall Ensemble Metrics
------------------------
  R2  Score              : {r2:.4f}
  Accuracy (1 - MAPE)    : {(1 - mape)*100:.2f}%
  MAPE                   : {mape*100:.2f}%

Individual Accuracies
---------------------
  XGBoost Accuracy       : {(1 - mape_xgb)*100:.2f}%
  LightGBM Accuracy      : {(1 - mape_lgb)*100:.2f}%
"""
    print(report)

    report_name = os.path.join(report_dir, 'xgb_lgbm_advanced_ensemble_optuna_performance.txt')
    with open(report_name, 'w', encoding='utf-8') as f:
        f.write(report)

    bundle = {
        'xgb'            : final_xgb,
        'lgb'            : final_lgb,
        'features'       : features,
        'label_encoders' : le_dict,
        'weights'        : {'xgb': optimal_w_xgb, 'lgb': optimal_w_lgb},
        'target_transform': 'log1p'
    }
    model_name = os.path.join(model_dir, 'xgb_lgbm_advanced_ensemble_optuna_model.joblib')
    joblib.dump(bundle, model_name)
    print(f"Artifacts saved explicitly to: {model_dir}")

if __name__ == "__main__":
    train_xgb_lgbm_advanced()

Loading data...


[I 2026-03-21 12:11:36,946] A new study created in memory with name: no-name-0774cc7d-dd3e-4c33-9865-acbbc8602968



--- Tuning XGBoost (50 Trials) ---


[I 2026-03-21 12:11:44,736] Trial 0 finished with value: 0.09454008662097307 and parameters: {'n_estimators': 398, 'learning_rate': 0.03298789834447851, 'max_depth': 8, 'subsample': 0.8600203803173458, 'colsample_bytree': 0.6810483379113468, 'min_child_weight': 4, 'reg_alpha': 0.27435540377739603, 'reg_lambda': 5.8085762034850345e-05}. Best is trial 0 with value: 0.09454008662097307.
[I 2026-03-21 12:11:51,282] Trial 1 finished with value: 0.10801808383544993 and parameters: {'n_estimators': 623, 'learning_rate': 0.08535407938473497, 'max_depth': 6, 'subsample': 0.863246389351331, 'colsample_bytree': 0.9777723011146737, 'min_child_weight': 6, 'reg_alpha': 3.3652403542598574e-08, 'reg_lambda': 0.0037686096836404975}. Best is trial 0 with value: 0.09454008662097307.
[I 2026-03-21 12:12:03,480] Trial 2 finished with value: 0.0992264104953541 and parameters: {'n_estimators': 970, 'learning_rate': 0.040938293043380404, 'max_depth': 7, 'subsample': 0.8452881811769203, 'colsample_bytree': 0.8

Best XGBoost Params (MAPE: 0.0921): {'n_estimators': 876, 'learning_rate': 0.00819190436989011, 'max_depth': 6, 'subsample': 0.7652822079106122, 'colsample_bytree': 0.7262048553028914, 'min_child_weight': 9, 'reg_alpha': 1.7391368145293053e-06, 'reg_lambda': 2.629441405815358e-08}

--- Tuning LightGBM (50 Trials) ---


[I 2026-03-21 12:22:05,089] Trial 0 finished with value: 0.0920327301935645 and parameters: {'n_estimators': 671, 'learning_rate': 0.010636833490397806, 'max_depth': 8, 'num_leaves': 103, 'subsample': 0.6878714098152299, 'colsample_bytree': 0.6527890246791828, 'min_child_samples': 26, 'reg_alpha': 1.5017683451492234e-05, 'reg_lambda': 1.2448223359732333e-07}. Best is trial 0 with value: 0.0920327301935645.
[I 2026-03-21 12:22:09,811] Trial 1 finished with value: 0.0921390306564554 and parameters: {'n_estimators': 553, 'learning_rate': 0.015001440734192852, 'max_depth': 6, 'num_leaves': 55, 'subsample': 0.9941676659252885, 'colsample_bytree': 0.6656665304947049, 'min_child_samples': 27, 'reg_alpha': 0.6130694928480235, 'reg_lambda': 0.01815310454971989}. Best is trial 0 with value: 0.0920327301935645.
[I 2026-03-21 12:22:11,543] Trial 2 finished with value: 0.09337291904010322 and parameters: {'n_estimators': 302, 'learning_rate': 0.03441030168385344, 'max_depth': 5, 'num_leaves': 29, '

Best LightGBM Params (MAPE: 0.0919): {'n_estimators': 581, 'learning_rate': 0.013923152508847651, 'max_depth': 6, 'num_leaves': 71, 'subsample': 0.6981169548458899, 'colsample_bytree': 0.7273246742744683, 'min_child_samples': 25, 'reg_alpha': 6.205688650478069e-05, 'reg_lambda': 0.08890721233944927}

Training Final Tuned Models...


[I 2026-03-21 12:28:11,207] A new study created in memory with name: no-name-1b8ae1f8-744d-457f-833a-248a9af14d2c
[I 2026-03-21 12:28:11,211] Trial 0 finished with value: 0.0918787589430392 and parameters: {'w_xgb': 0.7772037995184795}. Best is trial 0 with value: 0.0918787589430392.
[I 2026-03-21 12:28:11,214] Trial 1 finished with value: 0.09184762565406104 and parameters: {'w_xgb': 0.7348888289982288}. Best is trial 1 with value: 0.09184762565406104.
[I 2026-03-21 12:28:11,217] Trial 2 finished with value: 0.09196439304399925 and parameters: {'w_xgb': 0.8750652753224561}. Best is trial 1 with value: 0.09184762565406104.
[I 2026-03-21 12:28:11,221] Trial 3 finished with value: 0.09175377516691259 and parameters: {'w_xgb': 0.20427755517323143}. Best is trial 3 with value: 0.09175377516691259.
[I 2026-03-21 12:28:11,223] Trial 4 finished with value: 0.09204026725909614 and parameters: {'w_xgb': 0.9450632937910004}. Best is trial 3 with value: 0.09175377516691259.
[I 2026-03-21 12:28:11


--- Tuning Ensemble Weights ---
Optimal Ensemble Weight found -> XGBoost: 0.371, LightGBM: 0.629
Advanced XGBoost + LightGBM Ensemble - DEEP OPTUNA TUNED
Model built from: Final_Combined_data.csv 
Target  : np.log1p(retail_price) -> Expm1 inverted logic
Optuna Trials: 50 per algorithm + 30 for Weight blending

Tuned Hyperparameters
--------------------
  XGBoost  : {'n_estimators': 876, 'learning_rate': 0.00819190436989011, 'max_depth': 6, 'subsample': 0.7652822079106122, 'colsample_bytree': 0.7262048553028914, 'min_child_weight': 9, 'reg_alpha': 1.7391368145293053e-06, 'reg_lambda': 2.629441405815358e-08}
  LightGBM : {'n_estimators': 581, 'learning_rate': 0.013923152508847651, 'max_depth': 6, 'num_leaves': 71, 'subsample': 0.6981169548458899, 'colsample_bytree': 0.7273246742744683, 'min_child_samples': 25, 'reg_alpha': 6.205688650478069e-05, 'reg_lambda': 0.08890721233944927}

Ensemble Blending
-----------------
  XGBoost Weight : 0.3715
  LightGBM Weight: 0.6285

Overall Ensemble M